# 03 - Diagnostic Analysis

This notebook explains the main business results using simple comparisons that are easy to present in an interview. It focuses on category concentration, repeat-customer contribution, return drivers, payment mix, and seller risk.

In [ ]:
import pandas as pd

orders = pd.read_csv('../clean_csv/orders.csv', parse_dates=['order_date'])
products = pd.read_csv('../clean_csv/products.csv')
returns = pd.read_csv('../clean_csv/returns.csv')
transactions = pd.read_csv('../clean_csv/transactions.csv')
sellers = pd.read_csv('../clean_csv/sellers.csv')

order_products = orders.merge(products[['product_id', 'category']], on='product_id', how='left')
print(f'Orders: {len(orders):,} | Revenue: {orders.order_total_inr.sum():,.2f} INR')

## 1. Which categories drive revenue?

In [ ]:
category = (order_products.groupby('category', as_index=False)
    .agg(revenue_inr=('order_total_inr', 'sum'), units_sold=('quantity', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('revenue_inr', ascending=False))
category['revenue_share_pct'] = category['revenue_inr'] / category['revenue_inr'].sum() * 100
category.head(5)

Compare the top categories' revenue share with their order and unit volume. This shows whether a category is driven by demand, higher-value orders, or both.

## 2. Are repeat buyers driving the business?

In [ ]:
customer_value = (orders.groupby('customer_id', as_index=False)
    .agg(order_count=('order_id', 'nunique'), revenue_inr=('order_total_inr', 'sum')))
customer_value['buyer_segment'] = 'Repeat buyer'
customer_value.loc[customer_value['order_count'].eq(1), 'buyer_segment'] = 'One-time buyer'

segment = (customer_value.groupby('buyer_segment', as_index=False)
    .agg(customers=('customer_id', 'nunique'), orders=('order_count', 'sum'), revenue_inr=('revenue_inr', 'sum')))
segment['revenue_share_pct'] = segment['revenue_inr'] / segment['revenue_inr'].sum() * 100
segment

A large repeat-buyer revenue share supports retention, loyalty, and customer-lifetime-value recommendations.

## 3. What is driving returns?

In [ ]:
return_diagnostics = (returns.groupby('return_reason', as_index=False)
    .agg(return_count=('return_id', 'nunique'), refund_inr=('refund_amount_inr', 'sum'))
    .sort_values('return_count', ascending=False))
return_diagnostics['return_share_pct'] = return_diagnostics['return_count'] / return_diagnostics['return_count'].sum() * 100
return_diagnostics.head(10)

Prioritize the most frequent reasons first. Missing parts, late delivery, damaged packaging, and wrong-item issues point to fulfillment and quality-control actions.

## 4. Which payment methods matter most?

In [ ]:
payment_mix = (transactions.groupby('payment_method', as_index=False)
    .agg(transaction_count=('transaction_id', 'nunique'), payment_value_inr=('amount_inr', 'sum'))
    .sort_values('payment_value_inr', ascending=False))
payment_mix['value_share_pct'] = payment_mix['payment_value_inr'] / payment_mix['payment_value_inr'].sum() * 100
payment_mix

Use payment value share to understand customer payment preferences and where payment or refund monitoring matters most.

## 5. Which sellers need attention?

In [ ]:
seller_risk = sellers[['seller_id', 'seller_name', 'account_health', 'seller_rating', 'return_rate_pct', 'defect_rate_pct']].copy()
seller_risk['combined_risk_pct'] = seller_risk['return_rate_pct'] + seller_risk['defect_rate_pct']
seller_risk.sort_values(['combined_risk_pct', 'return_rate_pct'], ascending=False).head(10)

The diagnostic conclusion is actionable: protect high-value categories, retain repeat buyers, investigate leading return causes, monitor payment concentration, and coach sellers with elevated combined risk.